In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import math
import os

import numpy as np
import torch
from torch_geometric import seed_everything

seed_everything(42)

cache_dir = "./.cache_predql"

In [5]:
from experiments.model_training.utils import get_device

device = get_device()

Using device: cuda


In [6]:
import redelex

from relbench.datasets import get_dataset
from relbench.modeling.utils import get_stype_proposal
from predql_tasks.tasks import StatsUserEngagementTmpTask

In [7]:
dataset_name = "ctu-stats"
task_name = "stats_user_engagement_tmp"

dataset = get_dataset("ctu-stats", download=False)
task = StatsUserEngagementTmpTask()
db = dataset.get_db()

Loading Database object from /home/kolesiko/.cache/relbench/ctu-stats/db...
Done in 0.80 seconds.


In [8]:
col_to_stype_dict = get_stype_proposal(db)
col_to_stype_dict

{'postHistory': {'__PK__': <stype.numerical: 'numerical'>,
  'PostHistoryTypeId': <stype.numerical: 'numerical'>,
  'PostId': <stype.numerical: 'numerical'>,
  'RevisionGUID': <stype.text_embedded: 'text_embedded'>,
  'CreationDate': <stype.timestamp: 'timestamp'>,
  'UserId': <stype.numerical: 'numerical'>,
  'Text': <stype.text_embedded: 'text_embedded'>,
  'Comment': <stype.text_embedded: 'text_embedded'>,
  'UserDisplayName': <stype.text_embedded: 'text_embedded'>,
  'FK_posts_PostId': <stype.numerical: 'numerical'>,
  'FK_users_UserId': <stype.numerical: 'numerical'>},
 'votes': {'__PK__': <stype.numerical: 'numerical'>,
  'PostId': <stype.numerical: 'numerical'>,
  'VoteTypeId': <stype.numerical: 'numerical'>,
  'CreationDate': <stype.timestamp: 'timestamp'>,
  'UserId': <stype.numerical: 'numerical'>,
  'BountyAmount': <stype.numerical: 'numerical'>,
  'FK_posts_PostId': <stype.numerical: 'numerical'>,
  'FK_users_UserId': <stype.numerical: 'numerical'>},
 'tags': {'__PK__': <st

In [9]:
from experiments.model_training.training.text_embedder import TextEmbedder 
from experiments.model_training.utils import patched_to_unix_time

import relbench.modeling.graph
import relbench.modeling.utils
from relbench.modeling.graph import make_pkey_fkey_graph
from torch_frame.config.text_embedder import TextEmbedderConfig


text_embedder = TextEmbedderConfig(
    text_embedder=TextEmbedder(
        "BAAI/bge-base-en-v1.5",
        device=device,
        cache_dir=cache_dir), batch_size=256
)

relbench.modeling.graph.to_unix_time = patched_to_unix_time
relbench.modeling.utils.to_unix_time = patched_to_unix_time

data, col_stats_dict = make_pkey_fkey_graph(
    db, 
    col_to_stype_dict, 
    text_embedder,
    cache_dir=os.path.join(cache_dir, dataset_name)
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/relbench/modeling/graph.py:93: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or m

In [10]:
data

HeteroData(
  postHistory={
    tf=TensorFrame([266825, 8]),
    time=[266825],
  },
  votes={
    tf=TensorFrame([295778, 5]),
    time=[295778],
  },
  tags={ tf=TensorFrame([1032, 4]) },
  posts={
    tf=TensorFrame([80899, 20]),
    time=[80899],
  },
  badges={
    tf=TensorFrame([69012, 3]),
    time=[69012],
  },
  users={
    tf=TensorFrame([34149, 13]),
    time=[34149],
  },
  postLinks={
    tf=TensorFrame([9580, 4]),
    time=[9580],
  },
  comments={
    tf=TensorFrame([154876, 6]),
    time=[154876],
  },
  (postHistory, f2p_FK_posts_PostId, posts)={ edge_index=[2, 266825] },
  (posts, rev_f2p_FK_posts_PostId, postHistory)={ edge_index=[2, 266825] },
  (postHistory, f2p_FK_users_UserId, users)={ edge_index=[2, 247052] },
  (users, rev_f2p_FK_users_UserId, postHistory)={ edge_index=[2, 247052] },
  (votes, f2p_FK_posts_PostId, posts)={ edge_index=[2, 295705] },
  (posts, rev_f2p_FK_posts_PostId, votes)={ edge_index=[2, 295705] },
  (votes, f2p_FK_users_UserId, users)={ edg

In [11]:
from experiments.model_training.utils import make_loaders

loader_dict = make_loaders(
    data=data,
    task=task,
    batch_size=256,
    num_neighbors=[128, 64, 32],
    is_temporal=True
)

Loading Database object from /home/kolesiko/.cache/relbench/ctu-stats/db...
Done in 0.82 seconds.
Loading Database object from /home/kolesiko/.cache/relbench/ctu-stats/db...
Done in 0.76 seconds.


In [ ]:
from experiments.model_training.utils import compute_pos_weight

in_channels = 128
task_type = task.task_type
dropout = 0.4
is_temporal = True
epochs = 50

mlp_config = {
    "in_channels": 128,
    "hidden_channels": 64,
    "out_channels": 1,
    "layers": 2,
    "act": "relu",
    "norm": "layer_norm",
    "bias": True
}

pos_weight = compute_pos_weight(task)

Class weights computed: pos_weight=5.60


## SAGE

In [ ]:
from experiments.model_training.training.trainer import Trainer
from experiments.model_training.training.models.sage_model import SAGEModel
import gc


sage_config = {
    "channels": 128,
    "aggr": "mean",
    "layers": 3,
}

sage_model = SAGEModel(
    data=data,
    col_stats_dict=col_stats_dict,
    in_channels=in_channels,
    gnn_config=sage_config,
    mlp_config=mlp_config,
    task_type=task_type,
    dropout=dropout,
    is_temporal=is_temporal
).to(device)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))
optimizer = torch.optim.AdamW(sage_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

trainer = Trainer(
    task=task,
    model=sage_model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)

best_weights = trainer.train(
        loader_dict,
        num_epochs=epochs,
        tune_metric="average_precision",
        higher_is_better=True,
        patience=10
)

torch.save(best_weights, os.path.join(cache_dir, dataset_name, f"{sage_model.gnn_name}_{task_name}_best_weights.pt"))
    
print(trainer.evaluate(loader_dict["train"]))
print(trainer.evaluate(loader_dict["val"]))
print(trainer.evaluate(loader_dict["test"]))

del sage_model, optimizer, scheduler, trainer
torch.cuda.empty_cache() 
gc.collect()

/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Training: 100%|██████████| 245/245 [00:24<00:00, 10.15it/s]


Epoch 1/50, Train Loss: 1.1797


Evaluating: 100%|██████████| 51/51 [00:01<00:00, 29.26it/s]


Epoch 1/50, Eval Metrics: {'roc_auc': 0.6101137567515935, 'average_precision': 0.21110953290455406, 'f1': 0.0, 'loss': 1.0852125407009154}
New best model found with average_precision: 0.2111


Training: 100%|██████████| 245/245 [00:23<00:00, 10.33it/s]


...

Epoch 9/50, Train Loss: 1.1763


Evaluating: 100%|██████████| 51/51 [00:01<00:00, 26.27it/s]


Epoch 9/50, Eval Metrics: {'roc_auc': 0.4843046365283639, 'average_precision': 0.11763579633186881, 'f1': 0.0, 'loss': 1.0718475035311836}


Training:   2%|▏         | 4/245 [00:00<00:26,  9.14it/s]


KeyboardInterrupt: 

## HGT

In [ ]:
from experiments.model_training.training.trainer import Trainer
from experiments.model_training.training.models.hgt_model import HGTModel
import gc


hgt_config = {
    "channels": 128,
    "heads": 4,
    "layers": 3
}

hgt_model = HGTModel(
    data=data,
    col_stats_dict=col_stats_dict,
    in_channels=in_channels,
    gnn_config=hgt_config,
    mlp_config=mlp_config,
    task_type=task_type,
    dropout=dropout,
    is_temporal=is_temporal
).to(device)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))
optimizer = torch.optim.AdamW(hgt_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

trainer = Trainer(
    task=task,
    model=hgt_model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)

best_weights = trainer.train(
        loader_dict,
        num_epochs=epochs,
        tune_metric="average_precision",
        higher_is_better=True,
        patience=10
)

torch.save(best_weights, os.path.join(cache_dir, dataset_name, f"{hgt_model.gnn_name}_{task_name}_best_weights.pt"))
    
print(trainer.evaluate(loader_dict["train"]))
print(trainer.evaluate(loader_dict["val"]))
print(trainer.evaluate(loader_dict["test"]))

del hgt_model, optimizer, scheduler, trainer
torch.cuda.empty_cache() 
gc.collect()

/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Training: 100%|██████████| 245/245 [00:12<00:00, 20.14it/s]


Epoch 1/50, Train Loss: 0.8394


Evaluating: 100%|██████████| 51/51 [00:00<00:00, 55.55it/s]


Epoch 1/50, Eval Metrics: {'roc_auc': 0.8458332609689874, 'average_precision': 0.4912338883556456, 'f1': 0.479781691887869, 'loss': 0.7614591333311918}
New best model found with average_precision: 0.4912


Training: 100%|██████████| 245/245 [00:12<00:00, 19.04it/s]


...

Evaluating: 100%|██████████| 51/51 [00:00<00:00, 54.74it/s]


{'roc_auc': 0.8703036986864423, 'average_precision': 0.5260605069662154, 'f1': 0.4671474358974359, 'loss': 0.7435617583776227}


Evaluating: 100%|██████████| 58/58 [00:01<00:00, 52.87it/s]


{'roc_auc': 0.8763149102056084, 'average_precision': 0.49044341757311183, 'f1': 0.4304948417555517, 'loss': 0.7418782663455185}


0

## HAN

In [ ]:
from experiments.model_training.training.trainer import Trainer
from experiments.model_training.training.models.han_model import HANModel
import gc


han_config = {
    "channels": 128,
    "heads": 2,
    "dropout": 0.7,
    "layers": 3,
}

han_model = HANModel(
    data=data,
    col_stats_dict=col_stats_dict,
    in_channels=in_channels,
    gnn_config=han_config,
    mlp_config=mlp_config,
    task_type=task_type,
    dropout=dropout,
    is_temporal=is_temporal
).to(device)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))
optimizer = torch.optim.AdamW(han_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

trainer = Trainer(
    task=task,
    model=han_model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)

best_weights = trainer.train(
        loader_dict,
        num_epochs=epochs,
        tune_metric="average_precision",
        higher_is_better=True,
        patience=10
)

torch.save(best_weights, os.path.join(cache_dir, dataset_name, f"{han_model.gnn_name}_{task_name}_best_weights.pt"))
    
print(trainer.evaluate(loader_dict["train"]))
print(trainer.evaluate(loader_dict["val"]))
print(trainer.evaluate(loader_dict["test"]))

del han_model, optimizer, scheduler, trainer
torch.cuda.empty_cache() 
gc.collect()

/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Training: 100%|██████████| 245/245 [00:15<00:00, 15.78it/s]


Epoch 1/50, Train Loss: 1.1778


Evaluating: 100%|██████████| 51/51 [00:01<00:00, 41.34it/s]


Epoch 1/50, Eval Metrics: {'roc_auc': 0.5005985978684357, 'average_precision': 0.11955563712744588, 'f1': 0.0, 'loss': 1.0752883934565527}
New best model found with average_precision: 0.1196


Training: 100%|██████████| 245/245 [00:15<00:00, 16.19it/s]


...

Epoch 13/50, Train Loss: 1.1761


Evaluating: 100%|██████████| 51/51 [00:01<00:00, 39.17it/s]


Epoch 13/50, Eval Metrics: {'roc_auc': 0.4994014021315642, 'average_precision': 0.11921907947396328, 'f1': 0.0, 'loss': 1.0774029178664017}


Training:  85%|████████▍ | 208/245 [00:14<00:02, 14.85it/s]


KeyboardInterrupt: 

## GIN

In [ ]:
from experiments.model_training.training.trainer import Trainer
from experiments.model_training.training.models.gin_model import GINModel
import gc


gin_config = {
    "channels": 128,
    "mlp_layers": 2,
    "mlp_dropout": 0.3,
    "mlp_act": "relu",
    "mlp_norm": "batch_norm",
    "mlp_bias": True,
    "train_eps": True,
    "layers": 3,
}

gin_model = GINModel(
    data=data,
    col_stats_dict=col_stats_dict,
    in_channels=in_channels,
    gnn_config=gin_config,
    mlp_config=mlp_config,
    task_type=task_type,
    dropout=dropout,
    is_temporal=is_temporal
).to(device)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))
optimizer = torch.optim.AdamW(gin_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

trainer = Trainer(
    task=task,
    model=gin_model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)

best_weights = trainer.train(
        loader_dict,
        num_epochs=epochs,
        tune_metric="average_precision",
        higher_is_better=True,
        patience=10
)

torch.save(best_weights, os.path.join(cache_dir, dataset_name, f"{gin_model.gnn_name}_{task_name}_best_weights.pt"))
    
print(trainer.evaluate(loader_dict["train"]))
print(trainer.evaluate(loader_dict["val"]))
print(trainer.evaluate(loader_dict["test"]))

del gin_model, optimizer, scheduler, trainer
torch.cuda.empty_cache() 
gc.collect()

/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Training: 100%|██████████| 245/245 [00:15<00:00, 15.54it/s]


Epoch 1/50, Train Loss: 0.8533


Evaluating: 100%|██████████| 51/51 [00:01<00:00, 40.98it/s]


Epoch 1/50, Eval Metrics: {'roc_auc': 0.8644680641669127, 'average_precision': 0.5071122845851623, 'f1': 0.4933500627352572, 'loss': 0.717451859190014}
New best model found with average_precision: 0.5071


Training: 100%|██████████| 245/245 [00:15<00:00, 15.86it/s]


...

Evaluating: 100%|██████████| 51/51 [00:01<00:00, 40.86it/s]


{'roc_auc': 0.8628439766813132, 'average_precision': 0.5213435611670886, 'f1': 0.5021551724137931, 'loss': 0.7135099402827889}


Evaluating: 100%|██████████| 58/58 [00:01<00:00, 40.35it/s]


{'roc_auc': 0.8766446059678278, 'average_precision': 0.48838718762363253, 'f1': 0.4820366404948846, 'loss': 0.6568124360432106}


0

## GATv2

In [ ]:
from experiments.model_training.training.trainer import Trainer
from experiments.model_training.training.models.gatv2_model import GATv2Model
import gc


gatv2_config = {
    "channels": 128,
    "heads": 4,
    "add_self_loops": False,
    "layers": 3,
}

gatv2_model = GATv2Model(
    data=data,
    col_stats_dict=col_stats_dict,
    in_channels=in_channels,
    gnn_config=gatv2_config,
    mlp_config=mlp_config,
    task_type=task_type,
    dropout=dropout,
    is_temporal=is_temporal
).to(device)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))
optimizer = torch.optim.AdamW(gatv2_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

trainer = Trainer(
    task=task,
    model=gatv2_model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)

best_weights = trainer.train(
        loader_dict,
        num_epochs=epochs,
        tune_metric="average_precision",
        higher_is_better=True,
        patience=10
)

torch.save(best_weights, os.path.join(cache_dir, dataset_name, f"{gatv2_model.gnn_name}_{task_name}_best_weights.pt"))
    
print(trainer.evaluate(loader_dict["train"]))
print(trainer.evaluate(loader_dict["val"]))
print(trainer.evaluate(loader_dict["test"]))

del gatv2_model, optimizer, scheduler, trainer
torch.cuda.empty_cache() 
gc.collect()

/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Training: 100%|██████████| 245/245 [00:17<00:00, 13.97it/s]


Epoch 1/30, Train Loss: 1.1792


Evaluating: 100%|██████████| 51/51 [00:01<00:00, 37.14it/s]


Epoch 1/30, Eval Metrics: {'roc_auc': 0.4994014021315642, 'average_precision': 0.11921907947396328, 'f1': 0.0, 'loss': 1.0798215464385177}
New best model found with average_precision: 0.1192


Training:  26%|██▌       | 63/245 [00:04<00:13, 13.32it/s]


KeyboardInterrupt: 